Bibliotecas utilizadas

In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from datetime import datetime
import requests
import time

Configurações para acessar o navegador Chrome

In [12]:
def configurar_navegador():
    opcoes = Options()
    opcoes.add_argument('--ignore-certificate-errors') 
    opcoes.add_argument('--disable-blink-features=AutomationControlled') 
    
    driver = webdriver.Chrome(options=opcoes)
    return driver

driver = configurar_navegador()

Acessando o site da Embrapa

In [13]:
url = "https://agromet.cpact.embrapa.br/"
driver.get(url)

Tempo de espera (30 segundos)

In [14]:
wait = WebDriverWait(driver, 30)

Preparação para coleta dos dados

In [15]:
def acessar_frame_dados(driver):
    #Identificar o frame correto para acessar os dados
    driver.switch_to.default_content() 
    
    frames = driver.find_elements(By.TAG_NAME, "frame") + driver.find_elements(By.TAG_NAME, "iframe")
    
    for indice in range(len(frames)):
        driver.switch_to.default_content()
        driver.switch_to.frame(indice)
        
        try:
            wait_curto = WebDriverWait(driver, 3)
            wait_curto.until(EC.presence_of_element_located((By.ID, "lblRealV1")))
            return True 
            
        except:
            continue 

    return False

sucesso = acessar_frame_dados(driver)

Coleta dos dados da cidade de Pelotas

In [16]:
def extrair_dados_clima(driver):
    #Extrai os dados de clima do frame correto e monta o pacote para o n8n
    try:
        temperatura = driver.find_element(By.ID, "lblRealV1").text
        umidade = driver.find_element(By.ID, "lblRealV2").text
        sensacao = driver.find_element(By.ID, "lblRealV3").text 
        chuva = driver.find_element(By.ID, "lblAcumuladoV1").text

        print("🌤️ BOLETIM")
        print("-" * 25)
        print(f"Temperatura: {temperatura}")
        print(f"Umidade: {umidade}")
        print(f"Sensação Térmica: {sensacao}")
        print(f"Chuva Diária: {chuva}\n")

        # 2. Cria o pacote (dicionário) que será enviado ao n8n
        dados = {
            "cidade": "Pelotas",
            "temperatura": temperatura,
            "umidade": umidade,
            "sensacao": sensacao,
            "chuva": chuva
        }
        return dados

    except Exception as e:
        print(f"Erro ao extrair os dados: {e}")
        return None

if sucesso: 
    boletim = extrair_dados_clima(driver)
    print("📦 Pacote 'boletim' criado e pronto para envio!")

🌤️ BOLETIM
-------------------------
Temperatura: 5.9 °C
Umidade: 82 %
Sensação Térmica: 5.8 °C
Chuva Diária: 0.0 mm

📦 Pacote 'boletim' criado e pronto para envio!


Coleta dos dados da cidade de Morro Redondo

In [17]:
def extrair_clima_morro_redondo():
    #Buscando os dados de clima de Morro Redondo via API pública da Open-Meteo
    # Coordenadas: Latitude -31.59, Longitude -52.64
    url = "https://api.open-meteo.com/v1/forecast?latitude=-31.59&longitude=-52.64&current=temperature_2m,relative_humidity_2m,apparent_temperature,precipitation"
    
    try:
        resposta = requests.get(url).json()
        atual = resposta["current"]
        
        dados_mr = {
            "temperatura_mr": f"{atual['temperature_2m']} °C",
            "umidade_mr": f"{atual['relative_humidity_2m']} %",
            "sensacao_mr": f"{atual['apparent_temperature']} °C",
            "chuva_mr": f"{atual['precipitation']} mm"
        }
        return dados_mr
        
    except Exception as e:
        print(f"Erro ao buscar Morro Redondo: {e}")
        return None

Captura da imagem de satélite

In [18]:
print("🌐 Redirecionando o robô para a REDEMET...")
driver.get("https://redemet.decea.mil.br/")
driver.maximize_window()

wait = WebDriverWait(driver, 15)

elemento_mapa = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".leaflet-container")))

acoes = ActionChains(driver)

# Primeiro arrasto
acoes.click_and_hold(elemento_mapa).move_by_offset(0, -170).release().perform()
time.sleep(1)

# Segundo arrasto 
acoes.click_and_hold(elemento_mapa).move_by_offset(0, -170).release().perform()
time.sleep(2) 

# Clica na camada "Satélite"
btn_satelite = driver.find_element(By.XPATH, "//li[@title='Imagens Satélite']")
btn_satelite.click()
time.sleep(2)

# Clica no botão de Zoom [+] duas vezes
btn_zoom = driver.find_element(By.CSS_SELECTOR, ".leaflet-control-zoom-in")
btn_zoom.click()
time.sleep(1)
btn_zoom.click()

btn_fechar_menu = driver.find_element(By.CSS_SELECTOR, ".toggle-sidebar")
btn_fechar_menu.click()
time.sleep(1)

# Captura da imagem do satélite
print("⏳ Aguardando satélite carregar as nuvens...")
time.sleep(6) 
foto_binaria = driver.get_screenshot_as_png()
print("📸 Print do satélite capturado com sucesso!")

🌐 Redirecionando o robô para a REDEMET...
⏳ Aguardando satélite carregar as nuvens...
📸 Print do satélite capturado com sucesso!


Enviar dados coletados para o n8n

In [19]:
def enviar_para_n8n(dados_clima, caminho_imagem="satelite_rs.png"):
    url_webhook = "http://localhost:5678/webhook/boletim-clima" 
    print("🚀 Enviando dados e imagem de satélite para o n8n...")
    
    try:
        arquivos = {
            "imagem_satelite": ("satelite_rs.png", foto_binaria, "image/png")
        }
            
        resposta = requests.post(url_webhook, data=dados_clima, files=arquivos)
            
        if resposta.status_code == 200:
            print("✅ Sucesso Absoluto! O n8n recebeu o pacote e a imagem.")
        else:
            print(f"⚠️ Erro no n8n. Status: {resposta.status_code}")
            
    except Exception as e:
        print(f"❌ Erro de comunicação: {e}")

Juntando os dados das duas cidades

In [20]:
boletim_pelotas = boletim 

mr = extrair_clima_morro_redondo()

data_hoje = datetime.now().strftime("%d/%m/%Y")

pacote_final = {
    "data_hoje": data_hoje,

    "cidade": boletim_pelotas["cidade"],
    "temperatura_pel": boletim_pelotas["temperatura"],
    "umidade_pel": boletim_pelotas["umidade"],
    "sensacao_pel": boletim_pelotas["sensacao"],
    "chuva_pel": boletim_pelotas["chuva"],

    "temperatura_mr": mr["temperatura_mr"],
    "umidade_mr": mr["umidade_mr"],
    "sensacao_mr": mr["sensacao_mr"],
    "chuva_mr": mr["chuva_mr"]
}

enviar_para_n8n(pacote_final, foto_binaria)

🚀 Enviando dados e imagem de satélite para o n8n...
✅ Sucesso Absoluto! O n8n recebeu o pacote e a imagem.
